### Week 2 Exercise: 
In this Tutor your can select from a list of languages and it will teach you 6 sentences and after that show you the charges for further learning.


In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3


In [2]:
# Initialization

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    


OpenAI API Key exists and begins sk-proj-


In [3]:

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"

ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

MODEL = "gpt-4.1-mini"
ollama_model = "llama3.2:latest"

In [4]:
#Creating a DB
DB = "Course_Price.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS courses (language TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [6]:
def set_course_price(language,price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO courses (language,price) VALUES(?,?) ON CONFLICT (language) DO UPDATE SET price = ?',(language.lower(),price,price))
        conn.commit()
        

In [7]:
course_prices = {"Spanish":1000,"French":1500,"Japanese":2000,"Chinese":2500,"Hindi":3000}
for language, price  in course_prices.items():
    set_course_price(language,price) 

In [8]:
def get_course_price(language):
    print(f"DATABASE TOOL CALLED: getting price for course: {language}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM courses WHERE language = ?',(language.lower(),))
        result = cursor.fetchone()
        return f"Price for {language} course is ${result[0]}" if result else "This language course is not available."

In [9]:
get_course_price("Japanese")

DATABASE TOOL CALLED: getting price for course: Japanese


'Price for Japanese course is $2000.0'

In [26]:
system_message = """
You are a helpful assistant for a Language Tutor Website called Langgo.
Remember these important points as a language tutor-
1. Ask the user which Language they want to learn. 
2. If the user asks which languages you teach then mention the languages only currently available in the database.
3. After the users tells you the language then Ask the user to input the sentence they want to translate.
4. Teach only 5 sentences to the user and after you have translated 5 sentences for the user then inform the user of the language couse price if they want to continue."

Give short, courteous answers.
Always be accurate. If you don't know the answer, say so.
"""

In [11]:
price_function = {
    "name": "get_course_price",
    "description": "Get the price of a language course.",
    "parameters": {
        "type": "object",
        "properties": {
            "language": {
                "type": "string",
                "description": "The language that the customer wants to learn",
            },
        },
        "required": ["language"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_course_price',
   'description': 'Get the price of a language course.',
   'parameters': {'type': 'object',
    'properties': {'language': {'type': 'string',
      'description': 'The language that the customer wants to learn'}},
    'required': ['language'],
    'additionalProperties': False}}}]

In [12]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="coral",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [24]:
def chat(history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_message}] + history
    response = openai.chat.completions.create(model = MODEL,messages = messages,tools = tools)
    languages = []

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, languages = handle_tool_calls_and_return_language(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model = MODEL,messages=messages,tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant","content":reply}]

    voice = talker(reply)

    return history, voice



In [15]:
def handle_tool_calls_and_return_language(message):
    responses =[]
    languages= []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_course_price":
            arguments = json.loads(tool_call.function.arguments)
            language = arguments.get('language')
            languages.append(language)
            price_details = get_course_price(language)
            responses.append({
                "role":"tool",
                "content":price_details,
                "tool_call_id":tool_call.id
            })
    return responses,languages

In [17]:
def put_message_in_chatbot(message,history):
    return "", history + [{"role":"user","content":message}]

In [29]:
#Gradio

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=400, type="messages")
        #image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label='Chat with our Language Tutor')

    message.submit(put_message_in_chatbot, inputs=[message,chatbot],outputs = [message,chatbot]).then(
    chat,inputs = chatbot,outputs = [chatbot,audio_output]
    )

ui.launch(inbrowser=True)
    





* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
